# 1. 環境の準備

In [ ]:
!pip install -U llama-cpp-python huggingface_hub psutil

In [ ]:
import os
import time
import psutil
from llama_cpp import Llama
from huggingface_hub import list_repo_files
from huggingface_hub import hf_hub_download

# 2. 関数定義
## 1. モデルダウンロード

In [ ]:
MODEL_DIR = "/home/pi/models"   # ←共通パス

def download_gguf(repo_id, filename):
    os.makedirs(MODEL_DIR, exist_ok=True)

    filepath = os.path.join(MODEL_DIR, filename)

    if os.path.exists(filepath):
        return filepath

    return hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False
    )

##  2. 推論コード（共通テンプレート）

In [ ]:
def run_inference(gguf_path, prompt, n_ctx=2048, n_threads=4,max_tokens=100,):
    print(f"\n=== MODEL ===\n{gguf_path}")

    print("=== モデルロード ===")
    llm = Llama(
        model_path=gguf_path,
        n_ctx=n_ctx,
        n_threads=n_threads,
        n_gpu_layers=0,   # Raspberry Pi / CPU前提
        verbose=False,    # ログ抑制
    )
    
    start = time.perf_counter()
    print("=== 推論開始 ===")
    
    out = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=0.1,
        top_p=0.9,
        stop=["<|system|>", "<|user|>"],
    )
    
    elapsed = time.perf_counter() - start
    response = out["choices"][0]["text"].strip()
    
    print("=== 推論完了 ===")
    print(response)
    print("\n--- STATS ---")
    print(f"推論時間: {elapsed:.2f} 秒")

In [ ]:
import time
from llama_cpp import Llama

# =========================
# モデルロード
# =========================
def load_model(gguf_path, n_ctx=2048, n_threads=4):
    print(f"\n=== モデルロード: {gguf_path} ===")
    start = time.perf_counter()

    llm = Llama(
        model_path=gguf_path,
        n_ctx=n_ctx,
        n_threads=n_threads,
        n_gpu_layers=0,
        verbose=False,
    )

    elapsed = time.perf_counter() - start
    print(f"ロード時間: {elapsed:.2f} 秒")
    return llm


# =========================
# 推論実行
# =========================
def run_inference(llm, prompt, max_tokens=100):
    start = time.perf_counter()

    out = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=0.1,
        top_p=0.9,
        stop=["<|system|>", "<|user|>"],
    )

    elapsed = time.perf_counter() - start
    response = out["choices"][0]["text"].strip()

    # トークン数を取得（usageがあれば）
    try:
        tokens = out["usage"]["completion_tokens"]
    except:
        tokens = len(response.split())

    tps = tokens / elapsed if elapsed > 0 else 0

    print("=== 推論完了 ===")
    print(response)
    print("\n--- STATS ---")
    print(f"時間: {elapsed:.2f} 秒")
    print(f"トークン数: {tokens}")
    print(f"処理速度: {tps:.2f} tokens/sec")

    return {
        "response": response,
        "time": elapsed,
        "tokens": tokens,
        "tokens_per_sec": tps,
    }

## 3. プロンプト

In [ ]:
prompt = "Explain Raspberry Pi in simple terms."

In [ ]:
prompt_instruct = """
<|system|>
You are a teacher for children.
<|user|>
Explain what a Raspberry Pi is in a way that elementary school kids can understand.
Requirements:
- Use no more than 2 sentences.
- Avoid technical jargon as much as possible.
<|assistant|>
"""

## 4. 推論

### 4-1. Phi-2
[https://huggingface.co/TheBloke/phi-2-GGUF](https://huggingface.co/TheBloke/phi-2-GGUF)  



In [ ]:
repo_id = "TheBloke/phi-2-GGUF"
files = list_repo_files(repo_id)

for f in files:
    if f.endswith(".gguf"):
        print(f)

In [ ]:
# モデルを取得
gguf_path = download_gguf("TheBloke/phi-2-GGUF", "phi-2.Q4_K_M.gguf")

# 推論実行
llm = load_model(gguf_path, n_ctx=1024, n_threads=4)
result = run_inference(llm, prompt)

### 4-2. Phi-3 mini

In [ ]:
repo_id = "bartowski/Phi-3.5-mini-instruct-GGUF"
files = list_repo_files(repo_id)

for f in files:
    if f.endswith(".gguf"):
        print(f)

In [ ]:
# モデルを取得
gguf_path = download_gguf(
    "bartowski/Phi-3.5-mini-instruct-GGUF", 
    "Phi-3.5-mini-instruct-Q4_K_M.gguf"
)
# 推論実行
llm = load_model(gguf_path, n_ctx=1024, n_threads=4)
result = run_inference(llm, prompt)

In [ ]:
# instruct用のプロンプトを使用
result = run_inference(llm, prompt_instruct)

### 4-3. Qwen2.5

In [ ]:
repo_id = "tensorblock/Qwen2.5-0.5B-GGUF"

files = list_repo_files(repo_id)

print("=== 全ファイル一覧 ===")
for f in files:
    print(f)

In [ ]:
# モデルを取得
gguf_path = download_gguf(
    "tensorblock/Qwen2.5-0.5B-GGUF",
    "Qwen2.5-0.5B-Q3_K_M.gguf"
)
# 推論実行
llm = load_model(gguf_path, n_ctx=1024, n_threads=4)
result = run_inference(llm, prompt)

### 4-5. Granite 4.0 h-tiny
[https://huggingface.co/ibm-granite/granite-4.0-h-tiny-GGUF](hhttps://huggingface.co/ibm-granite/granite-4.0-h-tiny-GGUF)

In [ ]:
repo_id = "ibm-granite/granite-4.0-h-tiny-GGUF"
files = list_repo_files(repo_id)

for f in files:
    print(f)

In [ ]:
# モデルを取得
gguf_path = download_gguf(
    "ibm-granite/granite-4.0-h-tiny-GGUF",
    "granite-4.0-h-tiny-Q4_K_M.gguf"
)
# 推論実行
llm = load_model(gguf_path, n_ctx=1024, n_threads=4)
result = run_inference(llm, prompt)

### 4-6. Granite 4.0 h-350m
[https://huggingface.co/ibm-granite/granite-4.0-h-tiny-GGUF](hhttps://huggingface.co/ibm-granite/granite-4.0-h-tiny-GGUF)

In [ ]:
repo_id = "ibm-granite/granite-4.0-h-350m-GGUF"
files = list_repo_files(repo_id)
for f in files:
    print(f)

In [ ]:
# モデルを取得
gguf_path = download_gguf(
    "ibm-granite/granite-4.0-h-350m-GGUF",
    "granite-4.0-h-350m-Q8_0.gguf"
)
# 推論実行
llm = load_model(gguf_path, n_ctx=1024, n_threads=4)
result = run_inference(llm, prompt)